# Baseline evaluation ? Colab T4

Run zero-shot, few-shot, and RAG with the same 4-bit Qwen model. Inspect the five-item outputs before continuing to the full set.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime (T4) in Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil, time

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/farmer-llm')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/sarthakmehra02/stuff.git'
REPO = Path('/content/farmer-llm-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)


In [ ]:
%pip install -r requirements-colab.txt
import importlib.metadata as m
for name in ['torch','transformers','peft','trl','bitsandbytes','sentence-transformers','faiss-cpu']:
    try: print(f'{name}={m.version(name)}')
    except m.PackageNotFoundError: print(f'{name}=not installed')


In [ ]:
%%time
from src.rag import build_index
index, chunks = build_index()
print(f'Rebuilt index: {index.ntotal} vectors from {len(chunks)} chunks')


In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --limit 5 --out results
for condition in ['zero_shot','prompted','rag']:
    print('\n###', condition)
    with open(f'results/raw_{condition}.jsonl', encoding='utf-8') as f:
        for line in f: print(json.loads(line)['id'], json.loads(line)['answer'])


## Check these five examples

Review the answers above. Once you are satisfied, run the next cell for the full 60-question baseline evaluation.

In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --out results --resume
shutil.copy2('results/run_metadata.json', DRIVE_ROOT / 'run_metadata.json')
for path in Path('results').glob('raw_*.jsonl'):
    shutil.copy2(path, DRIVE_ROOT / path.name)
print('Copied baseline outputs and metadata to', DRIVE_ROOT)
